<a href="https://colab.research.google.com/github/ibtihalalf/Sdaia-Bootcamp/blob/main/C4/M3/Ex1_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with Docling, LangChain, and ChromaDB

### Install dependencies

In [ ]:
%pip install -q --progress-bar off --no-warn-conflicts \
    langchain-docling langchain-core langchain-huggingface \
    langchain_community langchain-openai langchain-chroma chromadb \
    langgraph langchain python-dotenv rich requests

  Preparing metadata (setup.py) ... done


In [ ]:
import json
import os
from textwrap import dedent

from rich import print

### Environment variables

In [ ]:
from dotenv import load_dotenv

def _get_env_from_colab_or_os(key: str) -> str | None:
    """Read an environment variable from Colab secrets first, then OS env."""
    try:
        from google.colab import userdata
        try:
            return userdata.get(key)
        except userdata.SecretNotFoundError:
            pass
    except ImportError:
        pass
    return os.getenv(key)

In [ ]:
load_dotenv()

False

In [ ]:
# Disable tokenizer parallel workers to avoid noisy notebook warnings.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 1. Document loading and chunking


This section loads and chunks the source document using Docling so we can build retrieval units for the RAG pipeline.

In [ ]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer
from langchain_docling.loader import ExportType


# Source document.
file_path = ["/content/iso27001.pdf"]

# Same as embedding model.
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"

# Export mode used by this lesson; DOC_CHUNKS gives retrieval-ready chunks directly.
export_type = ExportType.DOC_CHUNKS

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(embed_model_id)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

In [ ]:
from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker

loader = DoclingLoader(
    file_path=file_path,
    export_type=export_type,
    chunker=HybridChunker(tokenizer=tokenizer),
)

docs = loader.load()

[INFO] 2026-05-02 14:21:39,413 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-02 14:21:39,422 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-02 14:21:40,235 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-05-02 14:21:40,324 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-02 14:21:40,332 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-02 14:21:40,539 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-02 14:21:40,544 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-02 14:21:40,990 [

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (12556 > 512). Running this sequence through the model will result in indexing errors


numer of chunks

In [ ]:
(type(docs), len(docs))

(list, 138)

content of the first few chunks:

In [ ]:
print(docs[1].page_content)

Information security, cybersecurity and privacy protection - Information security management systems Requirements
Sécurité de l'information, cybersécurité et protection de la vie privée - Systèmes de management de la sécurité de 
l'information Exigences
Reference number ISO/IEC 27001:2022(E)

In [ ]:
print(docs[2].page_content)

©  ISO/IEC 2022
All rights reserved. Unless otherwise specified, or required in the context of its implementation, no part of this 
publication may be reproduced or utilized otherwise in any form or by any means, electronic or mechanical, 
including photocopying, or posting on the internet or an intranet, without prior written permission. Permission can
be requested from either ISO at the address below or ISO's member body in the country of the requester.
ISO copyright office CP 401 • Ch. de Blandonnet 8
CH-1214 Vernier, Geneva
Phone: +41 22 749 01 11
Email: copyright@iso.org
Website: www.iso.org
Published in Switzerland
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---

In [ ]:
print(docs[5].page_content)

ISO/IEC 27001:2022(E)

 ..................................................................................................................
...................................................................................................................
..., 2 = Foreword

## 2. Indexing

### 2.1 Embedding Model

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name=embed_model_id)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 2.2 Vector Store

In [ ]:
from pathlib import Path
import shutil
import tempfile
import uuid

persist_directory = Path(tempfile.gettempdir()) / f"chroma_db_docling_{uuid.uuid4().hex[:8]}"
if persist_directory.exists():
    # Best-effort cleanup on reruns.
    shutil.rmtree(persist_directory, ignore_errors=True)


In [ ]:
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

vectorstore = Chroma.from_documents(
    documents=filter_complex_metadata(docs),
    embedding=embedding,
    collection_name="docling_demo",
    persist_directory=str(persist_directory),
)

## 3. Retrieval and Generation

### 3.1 Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### 3.2 LLM

In [ ]:
from langchain_openai import ChatOpenAI

# OpenRouter chat model
openrouter_llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=_get_env_from_colab_or_os("OPENROUTER_API_KEY"),
)

### 3.3 Tasks

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.func import entrypoint, task

In [ ]:
@task
def retrieve(query: str):
    """Fetch top-k relevant documents from the vector store retriever."""
    return retriever.invoke(query)

In [ ]:
@task
def generate(query: str, passages: list[str], use_openrouter: bool = True) -> str:
    """Generate an answer from retrieved passages using a selected LangChain LLM."""
    system_text = dedent("""
        Role: You are a helpful and informative assistant. Answer using the reference passages below.
        Style and Tone: Respond in complete sentences for a non-technical audience in a friendly, conversational tone.
        Note: If a passage is irrelevant, you may ignore it.
    """)
    passages_serialized = "\n".join(f"<passage>{p}</passage>" for p in passages)
    user_text = dedent(f"""
        <question>{query}</question>
        <passages>
        {passages_serialized}
        </passages>
    """)

    if use_openrouter:
        msg = openrouter_llm.invoke([
            SystemMessage(system_text),
            HumanMessage(user_text)
        ])
        return msg.content

### 3.4 Workflow Entrypoint

In [ ]:
@entrypoint()
def rag_qa_workflow(query: str, use_openrouter: bool = True) -> dict:
    """Run retrieval and generation as a functional LangGraph RAG workflow."""
    docs = retrieve(query).result()
    answer = generate(query, [doc.page_content for doc in docs], use_openrouter).result()
    return {"question": query, "answer": answer, "context": docs}

## 4. Running the Workflow

This cell executes the functional RAG workflow end-to-end (retrieve then generate), prints a clipped final answer, and shows the source chunks used for grounding.

In [ ]:
question = "what is Information security risk assessment?"
d = rag_qa_workflow.invoke(question, use_openrouter=True)

In [ ]:
def clip_text(text: str, threshold: int = 100) -> str:
    """Return a truncated preview of long text for cleaner notebook output."""
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [ ]:
clipped_answer = clip_text(d["answer"], threshold=200)
print(f"Question:\n{d['question']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(d["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")

Question:
what is Information security risk assessment?

Answer:
Information security risk assessment is a systematic processthat an organization uses to identify, analyze, and 
prioritize the risks to its information assets. It involves comparing the results of a r...

Source 1:

text: "6.1.2 Information security risk assessment\nThe organization shall define and apply an information 
security risk assessment process that:"

source: /content/iso27001.pdf

Source 2:

text: "8.2 Information security risk assessment\nThe organization shall perform information security risk 
assessments at planned intervals or when significant changes are proposed or occur, taking account of the criteria 
established in 6.1.2 a).\nThe organization shall retain documented information of the results of the information 
security risk assessments..."

source: /content/iso27001.pdf

Source 3:

text: "6.1.2 Information security risk assessment\n- 1) compare the results of risk analysis with the risk 
criteria established in 6.1.2 a); and\n- 2) prioritize the analysed risks for risk treatment.\nThe organization 
shall retain documented information about the information security risk assessment process."

source: /content/iso27001.pdf

In [ ]:
print(d['answer'])

Information security risk assessment is a systematic processthat an organization uses to identify, analyze, and 
prioritize the risks to its information assets. It involves comparing the results of a risk analysis with the 
organization’s own risk criteria, and then ranking those risks so they can be treated appropriately. The 
organization also keeps documented evidence of the assessment process and its results, and it repeats the 
assessment on a regular schedule or whenever major changes occur. In short, it’s a way to understand which security
threats matter most and to plan how to address them.